# 02 — Seleção e votação de features

**Frente:** Dev 2 · **Critérios:** 3 (três métodos de seleção) e 4 (votação e eleição das features)

Este notebook roda a cada ciclo do pipeline, logo depois do `01_etl.ipynb`:

1. **Regra de votação**: lida do `config/config.yaml`, onde foi fixada **antes** do ciclo 1.
2. **Três métodos de seleção** (`src/features.py`), todos ajustados **só no treino**: SelectKBest (estatística F), importância por Random Forest e RFE com Regressão Logística. Cada um devolve o ranking completo e as top-k.
3. **Votação**: cada método vota nas suas top-k; é eleita a feature com ≥ 2 de 3 votos; desempate pela média dos ranks; mínimo de 5 features.
4. **Registro no MLflow** (run com `cycle_id` e `data_version`) e gravação de `artifacts/<cycle_id>/features_eleitas.json`, que o Dev 3 consome.

Nada de validação ou teste é usado aqui: a seleção faz parte do treinamento e, se olhasse para a validação, as métricas do Dev 3 ficariam otimistas.

In [ ]:
import os, sys, json
from pathlib import Path

# funciona tanto rodando de notebooks/ (nbconvert) quanto da raiz
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.ingestao import carregar_config
from src.preprocessamento import carregar_particoes, carregar_schema, ALVO
from src.features import (carregar_regra_votacao, preparar_treino, executar_metodos,
                          votar, resumir_votacao, registrar_votacao, carregar_features_eleitas)

cfg = carregar_config()

# cycle_id: vem do workflow (variável CYCLE_ID); fallback local — mesma convenção do notebook 01
try:
    from src.rastreio import obter_cycle_id  # helper do Dev 4, quando existir
    CYCLE_ID = obter_cycle_id()
except ImportError:
    from datetime import datetime
    CYCLE_ID = os.getenv("CYCLE_ID") or datetime.now().strftime("%Y%m%d-%H%M") + "-local"

schema = carregar_schema(cfg)
DATA_VERSION = schema["data_version"]
FEATURES = schema["features"]
print("cycle_id:    ", CYCLE_ID)
print("data_version:", DATA_VERSION)
print("features candidatas:", len(FEATURES))
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)

## 1. Regra de votação (fixada antes de ver qualquer resultado)

A regra está em `config.yaml` → `votacao` e é lida por `carregar_regra_votacao`, que também valida os valores. O teste `tests/test_regra_votacao.py` trava esses números: se alguém mudar a regra depois de ver resultados, a suíte falha e a mudança aparece na revisão.

In [ ]:
regra = carregar_regra_votacao(cfg)
pd.Series({k: (", ".join(v) if isinstance(v, list) else v) for k, v in regra.items()}, name="valor").to_frame()

**Por que essa regra.**
- **top-k = 10** por método (≈ 26% das 38 candidatas): grande o bastante para os métodos se sobreporem, pequeno o bastante para cada voto significar algo.
- **≥ 2 de 3 votos**: maioria simples. Uma feature só entra se dois critérios diferentes (estatístico, de árvore e linear) concordarem, o que reduz a chance de escolher ruído que um método isolado valoriza.
- **Desempate pela média dos ranks**: usa a informação completa dos rankings (não só quem está no top-k) e é determinística.
- **Mínimo de 5 features**: se os métodos discordarem muito, os modelos do Dev 3 ainda recebem um conjunto utilizável; as que completam o mínimo ficam marcadas (`motivo = completou_minimo`).

## 2. Treino pré-processado (só o treino)

O pré-processador do Dev 1 (log com sinal → imputação pela mediana → padronização) é ajustado **apenas no treino**. Os três métodos recebem esse `X` e o alvo `flag`.

In [ ]:
treino, validacao, teste = carregar_particoes(cfg)
X, y = preparar_treino(treino, FEATURES, cfg)
print(f"treino: {X.shape[0]} carteiras × {X.shape[1]} features | fraude = {y.mean():.1%}")
print("validação e teste NÃO são usados neste notebook:", len(validacao), "e", len(teste), "linhas reservadas")
X.describe().T[["mean", "std", "min", "max"]].round(2).head(8)

## 3. Três métodos de seleção

| Método | O que mede | Enxerga |
|---|---|---|
| `selectkbest_f_classif` | estatística F (ANOVA) de cada feature isolada contra o alvo | relações lineares/univariadas |
| `random_forest` | redução média de impureza (Gini) numa floresta balanceada, seed 42 | não linearidades e interações |
| `rfe_logistic` | ordem de eliminação recursiva com Regressão Logística balanceada (1 por vez) | contribuição conjunta num modelo linear |

Os três olham para o problema de ângulos diferentes; é por isso que a votação faz sentido.

In [ ]:
resultados = executar_metodos(X, y, cfg)
for nome, r in resultados.items():
    print(f"{nome:24s} params: {r['params']}")

top = pd.DataFrame({nome: r["top_k"] for nome, r in resultados.items()},
                   index=pd.RangeIndex(1, regra["top_k"] + 1, name="rank"))
top

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (nome, r) in zip(axes, resultados.items()):
    rk = r["ranking"].head(15).iloc[::-1]
    cores = ["tab:orange" if f in r["top_k"] else "lightgray" for f in rk["feature"]]
    ax.barh(rk["feature"], rk["score"], color=cores)
    ax.set_title(nome, fontsize=10); ax.tick_params(axis="y", labelsize=7)
fig_rankings = fig
fig.suptitle("Top-15 de cada método (laranja = top-k que vira voto)")
plt.tight_layout(); plt.show()

**Quanto os métodos concordam?** Tamanho da interseção dos top-k, par a par:

In [ ]:
nomes = list(resultados)
conc = pd.DataFrame(index=nomes, columns=nomes, dtype=float)
for a in nomes:
    for b in nomes:
        conc.loc[a, b] = len(set(resultados[a]["top_k"]) & set(resultados[b]["top_k"]))
conc.astype(int)

## 4. Votação

`votar` aplica a regra e devolve a **tabela de votos** (feature × método, total de votos, rank em cada método, rank médio, eleita, motivo) e a lista de eleitas, em ordem.

In [ ]:
tabela, eleitas = votar(resultados, regra)
resumo = resumir_votacao(tabela, regra)
print(f"{resumo['n_eleitas']} features eleitas de {resumo['n_features_avaliadas']} "
      f"({resumo['n_eleitas_por_votos']} por votos, {resumo['n_completadas_pelo_minimo']} completadas pelo mínimo)")
print("features com 3 / 2 / 1 / 0 votos:", resumo["n_com_3_votos"], "/", resumo["n_com_2_votos"],
      "/", resumo["n_com_1_voto"], "/", resumo["n_sem_votos"])

votos_cols = [c for c in tabela.columns if c.startswith("voto_")]
tabela.loc[tabela["total_votos"] > 0, ["feature"] + votos_cols + ["total_votos", "rank_medio", "eleita", "motivo"]] \
      .style.apply(lambda s: ["background-color: #d9f2d9" if v else "" for v in tabela.loc[s.index, "eleita"]], axis=0)

In [ ]:
com_voto = tabela.loc[tabela["total_votos"] > 0].iloc[::-1]
fig_votos, ax = plt.subplots(figsize=(9, 0.32 * len(com_voto) + 1))
ax.barh(com_voto["feature"], com_voto["total_votos"],
        color=["tab:green" if e else "lightgray" for e in com_voto["eleita"]])
ax.axvline(regra["votos_minimos"] - 0.5, ls="--", color="k", lw=1)
ax.set_xlabel("votos"); ax.set_xticks([0, 1, 2, 3]); ax.tick_params(axis="y", labelsize=7)
ax.set_title(f"Votos por feature — eleitas em verde (≥ {regra['votos_minimos']} votos)")
plt.tight_layout(); plt.show()

print("Eleitas, em ordem:")
for i, f in enumerate(eleitas, 1):
    print(f"  {i:2d}. {f}")

## 5. Registro no MLflow e artefatos do ciclo

Um run por ciclo (etapa `votacao_features`) com tags `cycle_id` e `data_version`, parâmetros da regra e de cada método, métricas da votação e os arquivos como artefatos. Os mesmos arquivos ficam em `artifacts/<cycle_id>/` para o Dev 3 (`features_eleitas.json`) e para o painel do Dev 5 (`tabela_votos.csv`).

In [ ]:
registro = registrar_votacao(resultados, tabela, eleitas, regra, DATA_VERSION, CYCLE_ID, cfg)

import mlflow
with mlflow.start_run(run_id=registro["run_id"]):   # reabre o run do ciclo só para anexar as figuras
    mlflow.log_figure(fig_rankings, "figuras/rankings_top15.png")
    mlflow.log_figure(fig_votos, "figuras/votos_por_feature.png")

print("run_id:  ", registro["run_id"])
print("pasta:   ", registro["dir"].relative_to(RAIZ).as_posix())
print("arquivos:", sorted(p.relative_to(registro["dir"]).as_posix() for p in registro["dir"].rglob("*") if p.is_file()))

In [ ]:
eleitas_json = carregar_features_eleitas(CYCLE_ID, cfg)
print(json.dumps({k: eleitas_json[k] for k in ("cycle_id", "data_version", "mlflow_run_id", "n_eleitas", "features", "motivo")},
                 indent=2, ensure_ascii=False))
assert eleitas_json["features"] == eleitas
assert len(eleitas) >= regra["minimo_features"]

### Entregas para as outras frentes
- **Dev 3 (modelos):** `from src.features import carregar_features_eleitas` → `carregar_features_eleitas(CYCLE_ID)["features"]`. Os dois modelos devem usar exatamente essa lista, selecionando por nome na saída do pré-processador.
- **Dev 5 (painel):** `artifacts/<cycle_id>/tabela_votos.csv` com as colunas `feature`, `voto_<método>`, `total_votos`, `rank_<método>`, `rank_medio`, `eleita`, `motivo`.
- **Dev 4 (workflow):** este notebook lê `CYCLE_ID` do ambiente e grava no mesmo `mlflow.db` do notebook 01; a pasta `artifacts/<cycle_id>/` deve ser persistida junto com `mlartifacts/`.
- **Relatório:** a seção "Votação das features" (`docs/votacao_features.md`) compara as tabelas de votos dos ciclos 1 e 2.